##### **** This notebook apply doc_id transform to jsonl files and generate the correspoiding parquet files. 
##### **** In addition, it converts the parquet files to jsonl files

##### **** These uv pip installs need to be adapted to use the appropriate release level. Alternatively, The venv running the jupyter lab could be pre-configured with a requirement file that includes the right release. Example for transform developers working from git clone:
```
make venv 
source venv/bin/activate 
uv pip install jupyterlab
```

In [ ]:
## This is here as a reference only
# Users and application developers must use the right tag for the latest from pypi
# uv pip install data-prep-kit-transforms[doc_id]

##### ***** Import required classes and modules

In [ ]:
from dpk_doc_id import DocID
from dotenv import load_dotenv
import os
# Load environment variables from a .env file if present
load_dotenv()

INPUT_FOLDER=os.environ['INPUT_FOLDER']
OUTPUT_FOLDER=os.environ['OUTPUT_FOLDER']
OUTPUT_JSON=os.environ['OUTPUT_JSON']
display(f"Input folder: {INPUT_FOLDER}", f"Output folder: {OUTPUT_FOLDER}", f"Output JSON: {OUTPUT_JSON}")

##### ***** Setup runtime parameters for this transform
* doc_column - specifies name of the column containing the document (required for ID generation)
* hash_column - specifies name of the column created to hold the string document id, if None, id is not generated
##### ***** Use python runtime to invoke the transform

In [ ]:
DocID(input_folder= INPUT_FOLDER,
        output_folder=OUTPUT_FOLDER,
        doc_id_doc_column="messages",
        doc_id_hash_column="uuid",
        data_files_to_use="['.jsonl']"
        ).transform()

##### **** The specified folder will include the transformed parquet files.

In [ ]:
if x: 
    print ("No X")

In [ ]:
import glob
output_files=glob.glob(f"{OUTPUT_FOLDER}/**/*.parquet", recursive=True)
len(output_files)

In [ ]:
%%time
def write_parquet_to_jsonl(parquet_file_path, output_jsonl_path):
    import polars as pl
    import json
    try:
        df = pl.read_parquet(parquet_file_path)
    except Exception as e:
        print (f"Error reading {parquet_file_path}: {type(e)}-{str(e)}")
        raise e
    try:
        df.write_ndjson(output_jsonl_path)
    except Exception as e:
        print (f"Error writing {output_jsonl_path}: {type(e)}-{str(e)}")
        raise e



def write_parquet_folder_to_jsonl(parquet_folder, jsonl_folder):
    import os
    import glob
    parquet_files = glob.glob(f"{parquet_folder}/**/*.parquet", recursive=True)
    for parquet_file in parquet_files:
        json_file=f"{os.path.splitext(parquet_file)[0]}.jsonl"
        json_file=json_file.replace(parquet_folder, jsonl_folder)
        os.makedirs(os.path.dirname(json_file), exist_ok=True)
        print (f"Processing: {parquet_file} -> {json_file}")
        try:
            write_parquet_to_jsonl(parquet_file, json_file)
        except Exception as e:
            print (f"Error processing {parquet_file}: {type(e)}-{str(e)}")
            return

write_parquet_folder_to_jsonl(OUTPUT_FOLDER, OUTPUT_JSON)







In [ ]:
import glob
json_files=glob.glob(f"{OUTPUT_JSON}/**/*.jsonl", recursive=True)
len(json_files)